# Phase 4C Diagnostic: Treatment Diversity Analysis
## Why does 100% of portfolio selection consist of Road Diet (TRT_002)?

**Contract & Governance Context**:
- **Decision References**: D001 (Corridor grain), D004 (Analytical scope), D005 (Governance authority), D020 (CMF parameters), D021 (Economic valuation), D023 (BCR eligibility filter).
- **Objective**: Execute a read-only diagnostic to determine the analytical and structural mechanisms causing 100% of project candidate selections across all 36 planning portfolios to select **TRT_002 (Road Diet)**.
- **Data Sources**: `data/processed/corridor_treatment_benefits.parquet` (387 candidate rows) and `data/processed/portfolio_project_selections.parquet` (1,410 detail selection rows).


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BENEFITS_PATH = ROOT / "data" / "processed" / "corridor_treatment_benefits.parquet"
SELECTIONS_PATH = ROOT / "data" / "processed" / "portfolio_project_selections.parquet"
EVIDENCE_PATH = ROOT / "docs" / "evidence" / "treatment_cmf_evidence_matrix.csv"

df_benefits = pd.read_parquet(BENEFITS_PATH)
df_selections = pd.read_parquet(SELECTIONS_PATH)

print(f"Loaded treatment benefits panel: {len(df_benefits)} rows across {df_benefits['corridor_id'].nunique()} corridors.")
print(f"Loaded portfolio selections: {len(df_selections)} detail rows across {df_selections['portfolio_id'].nunique()} portfolios.")


Loaded treatment benefits panel: 387 rows across 43 corridors.
Loaded portfolio selections: 1410 detail rows across 36 portfolios.


## Part 1 — Per-Treatment Summary Statistics
Compare candidate treatment performance metrics (`benefit_cost_ratio`, `capital_project_cost`, `crashes_averted_total`, `present_value_benefit`) across candidate treatments (`TRT_001`, `TRT_002`, `TRT_004`) and uncertainty scenarios (`CONSERVATIVE`, `BASE`, `OPTIMISTIC`).


In [2]:
# 1. Summary by Treatment and Scenario Level
t_scen_summary = df_benefits.groupby(["treatment_id", "treatment_name", "scenario_level"]).agg(
    candidate_rows=("corridor_id", "count"),
    mean_bcr=("benefit_cost_ratio", "mean"),
    median_bcr=("benefit_cost_ratio", "median"),
    mean_capital_cost=("capital_project_cost", "mean"),
    mean_crashes_averted=("crashes_averted_total", "mean"),
    mean_pv_benefit=("present_value_benefit", "mean"),
).reset_index()

print("=== Summary by Treatment and Scenario Level ===")
display(t_scen_summary)

# Overall Summary across all scenarios
t_overall_summary = df_benefits.groupby(["treatment_id", "treatment_name"]).agg(
    candidate_rows=("corridor_id", "count"),
    mean_bcr=("benefit_cost_ratio", "mean"),
    median_bcr=("benefit_cost_ratio", "median"),
    mean_capital_cost=("capital_project_cost", "mean"),
    mean_crashes_averted=("crashes_averted_total", "mean"),
    mean_pv_benefit=("present_value_benefit", "mean"),
).reset_index()

print("\n=== Overall Per-Treatment Summary Across All Scenarios ===")
display(t_overall_summary)


=== Summary by Treatment and Scenario Level ===


,treatment_id,treatment_name,scenario_level,candidate_rows,mean_bcr,median_bcr,mean_capital_cost,mean_crashes_averted,mean_pv_benefit
0,TRT_001,Pedestrian Refuge Islands & Medians,BASE,43,67.136639,59.543867,144930.232558,1.703527,8.796360e+06
1,TRT_001,Pedestrian Refuge Islands & Medians,CONSERVATIVE,43,20.047624,16.886473,179837.209302,0.669167,3.455320e+06
2,TRT_001,Pedestrian Refuge Islands & Medians,OPTIMISTIC,43,196.794360,169.061057,87565.116279,3.103082,1.602312e+07
3,TRT_002,Road Diet (4-to-3 Lane Conversion),BASE,43,811.313978,800.082883,155907.762907,70.964541,1.210373e+08
4,TRT_002,Road Diet (4-to-3 Lane Conversion),CONSERVATIVE,43,320.998334,316.554724,216538.559593,38.996239,6.651207e+07
5,TRT_002,Road Diet (4-to-3 Lane Conversion),OPTIMISTIC,43,2861.644282,2822.030273,69292.339070,111.246298,1.897419e+08
6,TRT_004,RRFB at Uncontrolled Marked Pedestrian Crossing,BASE,43,63.422287,56.249587,103358.139535,2.001645,5.926128e+06
7,TRT_004,RRFB at Uncontrolled Marked Pedestrian Crossing,CONSERVATIVE,43,22.460556,18.918929,119854.651163,0.871439,2.580009e+06
8,TRT_004,RRFB at Uncontrolled Marked Pedestrian Crossing,OPTIMISTIC,43,144.101085,125.992318,76454.651163,3.390616,1.003836e+07



=== Overall Per-Treatment Summary Across All Scenarios ===


,treatment_id,treatment_name,candidate_rows,mean_bcr,median_bcr,mean_capital_cost,mean_crashes_averted,mean_pv_benefit
0,TRT_001,Pedestrian Refuge Islands & Medians,129,94.659541,58.784318,137444.186047,1.825259,9.424934e+06
1,TRT_002,Road Diet (4-to-3 Lane Conversion),129,1331.318864,800.082883,147246.220523,73.735693,1.257638e+08
2,TRT_004,RRFB at Uncontrolled Marked Pedestrian Crossing,129,76.661309,52.202412,99889.147287,2.087900,6.181498e+06


## Part 2 — Highest BCR Analysis Per Corridor
Determine which candidate treatment achieves the highest Benefit-Cost Ratio (BCR) for each of the 43 high-crash corridors, broken down by scenario level.


In [3]:
# Identify highest BCR treatment per corridor & scenario
idx_max_bcr = df_benefits.groupby(["corridor_id", "scenario_level"])["benefit_cost_ratio"].idxmax()
df_best_bcr = df_benefits.loc[idx_max_bcr]

bcr_win_counts = df_best_bcr.groupby(["scenario_level", "treatment_id", "treatment_name"]).size().reset_index(name="corridor_win_count")
bcr_win_counts["win_percentage"] = (bcr_win_counts["corridor_win_count"] / 43.0) * 100.0

print("=== Highest BCR Winner Frequency per Scenario (43 Corridors) ===")
display(bcr_win_counts)


=== Highest BCR Winner Frequency per Scenario (43 Corridors) ===


,scenario_level,treatment_id,treatment_name,corridor_win_count,win_percentage
0,BASE,TRT_002,Road Diet (4-to-3 Lane Conversion),43,100.0
1,CONSERVATIVE,TRT_002,Road Diet (4-to-3 Lane Conversion),43,100.0
2,OPTIMISTIC,TRT_002,Road Diet (4-to-3 Lane Conversion),43,100.0


## Part 3 — Dominance Decomposition & Case Studies
Decompose why `TRT_002 (Road Diet)` wins across corridors. Compare candidate treatments on specific representative corridors to evaluate:
1. Crashes Averted (Target scope signal: All crashes vs Pedestrian-only crashes)
2. Capital Project Cost (Provisional cost signal)
3. Benefit-Cost Ratio (Combined economic return)


In [4]:
sample_corridors = ["HCC001", "HCC002", "HCC010", "HCC025"]
df_base = df_benefits[df_benefits["scenario_level"] == "BASE"].copy()

sample_comp = df_base[df_base["corridor_id"].isin(sample_corridors)][[
    "corridor_id",
    "corridor_name",
    "treatment_id",
    "treatment_name",
    "crashes_averted_total",
    "capital_project_cost",
    "present_value_benefit",
    "benefit_cost_ratio",
]].sort_values(["corridor_id", "treatment_id"])

print("=== Representative Corridor Treatment Comparison (BASE Scenario) ===")
display(sample_comp)


=== Representative Corridor Treatment Comparison (BASE Scenario) ===


,corridor_id,corridor_name,treatment_id,treatment_name,crashes_averted_total,capital_project_cost,present_value_benefit,benefit_cost_ratio
0,HCC001,Devon,TRT_001,Pedestrian Refuge Islands & Medians,2.952052,114000.000,1.567966e+07,137.540910
3,HCC001,Devon,TRT_002,Road Diet (4-to-3 Lane Conversion),92.037017,146094.165,1.312162e+08,898.161928
6,HCC001,Devon,TRT_004,RRFB at Uncontrolled Marked Pedestrian Crossing,3.468661,81300.000,1.056342e+07,129.931424
9,HCC002,Broadway,TRT_001,Pedestrian Refuge Islands & Medians,1.714037,152000.000,8.399469e+06,55.259667
12,HCC002,Broadway,TRT_002,Road Diet (4-to-3 Lane Conversion),60.165932,180711.090,7.984405e+07,441.832617
15,HCC002,Broadway,TRT_004,RRFB at Uncontrolled Marked Pedestrian Crossing,2.013993,108400.000,5.658741e+06,52.202412
81,HCC010,Ashland,TRT_001,Pedestrian Refuge Islands & Medians,2.001259,266000.000,9.153317e+06,34.410968
84,HCC010,Ashland,TRT_002,Road Diet (4-to-3 Lane Conversion),128.597419,329568.750,1.804486e+08,547.529327
87,HCC010,Ashland,TRT_004,RRFB at Uncontrolled Marked Pedestrian Crossing,2.351479,189700.000,6.166611e+06,32.507172
216,HCC025,Stony Island,TRT_001,Pedestrian Refuge Islands & Medians,1.727972,190000.000,8.960586e+06,47.160977


## Part 4 — Pure CMF Effect Analysis (Crashes Averted Alone)
Isolate the safety effect (CMF & target crash scope) from the capital cost signal by determining which treatment averts the highest number of annual crashes if costs were equalized or ignored.


In [5]:
# Identify treatment with highest crashes_averted_total per corridor & scenario
idx_max_averted = df_benefits.groupby(["corridor_id", "scenario_level"])["crashes_averted_total"].idxmax()
df_best_averted = df_benefits.loc[idx_max_averted]

averted_win_counts = df_best_averted.groupby(["scenario_level", "treatment_id", "treatment_name"]).size().reset_index(name="corridor_win_count")
averted_win_counts["win_percentage"] = (averted_win_counts["corridor_win_count"] / 43.0) * 100.0

print("=== Highest Crashes Averted Winner Frequency per Scenario (43 Corridors) ===")
display(averted_win_counts)


=== Highest Crashes Averted Winner Frequency per Scenario (43 Corridors) ===


,scenario_level,treatment_id,treatment_name,corridor_win_count,win_percentage
0,BASE,TRT_002,Road Diet (4-to-3 Lane Conversion),43,100.0
1,CONSERVATIVE,TRT_002,Road Diet (4-to-3 Lane Conversion),43,100.0
2,OPTIMISTIC,TRT_002,Road Diet (4-to-3 Lane Conversion),43,100.0


## Part 5 — Diagnostic Findings & Guidance for Phase GB2

### 1. Root Mechanism of Road Diet (TRT_002) Dominance
- **Target Crash Scope**: `TRT_002 (Road Diet)` is an **All-Crash** treatment (`target: total`). It applies its CMF (0.71 point CMF, 29% reduction) to all 2026 forecast corridor crashes (mean ~244 crashes/yr per corridor). By contrast, `TRT_001` (Refuge Islands) and `TRT_004` (RRFBs) are **Pedestrian-Only** treatments (`target: pedestrian`), targeting only ~2.5% of total corridor crashes (mean ~3–4 ped crashes/yr).
- **Magnitude of Safety Benefits**: Under BASE scenario, TRT_002 averts **~71.0 crashes/yr** per corridor on average, compared to **~1.70 crashes/yr** for TRT_001 and **~2.00 crashes/yr** for TRT_004.
- **Cost Equivalence Under Provisional Unit Costs**: Mean capital cost for TRT_002 ($155,908) is nearly identical to TRT_001 ($144,930) and only slightly higher than TRT_004 ($103,358).
- **Economic Return (BCR)**: Because TRT_002 delivers ~40x more averted crashes for ~1.1x–1.5x the cost, its mean BCR (811.3 in BASE scenario) is **>12x higher** than TRT_001 (67.1) or TRT_004 (63.4). TRT_002 achieves the highest BCR on **100% of corridors (43/43)** across all scenarios.
- **Pure Safety Signal**: Even if costs were completely equalized, TRT_002 averts the most crashes on **100% of corridors (43/43)** because all-crash reductions dominate pedestrian-only reductions in volume.

### 2. Implications & Strategic Direction for Phase GB2 (Realistic Costs & Applicability)
- **Realistic Unit Costs**: In Phase GB2, introducing realistic per-mile Road Diet costs ($250k–$1M+/mile including re-striping, signal heads, and access management) will raise TRT_002 capital costs significantly, lowering its BCR and making budgets binding.
- **Physical Applicability Filters**: TRT_002 is legally/physically constrained to 4-lane undivided arterials with ADT <= 20,000 vpd. Enforcing physical applicability screening will disqualify TRT_002 on multi-lane divided or high-volume corridors, mandating selection of TRT_001 or TRT_004.
- **Multi-Objective & Targeted Safety Floor**: Incorporating pedestrian-specific safety objectives or treatment diversity constraints in portfolio optimization will ensure location-specific pedestrian countermeasures (refuge islands, RRFBs) are programmed alongside corridor-level road diets.
